# 06 — Multimodal fusion training

## Architecture, as actually implemented

Three encoders, each producing a **fixed-width embedding**:

| Encoder | Class | Signature |
|---|---|---|
| HTML | `HTMLModalityModel` | `(token_ids, token_mask, dom_feats)` |
| Vision | `VisionModalityModel` | `(image)` |
| Fusion | `FusionModel` | `(emb_url, emb_html, emb_vision, mask_url, mask_html, mask_vision)` |

Two-stage training, matching `training/train_multimodal.py`:

1. Train HTML and vision encoders **standalone** on their own labels.
2. **Freeze** them, precompute embeddings for every row, then train only the fusion
   head on cached embeddings.

## Why fusion is not concatenation

Gating weights are re-normalised over the **available** modalities only, so a missing
modality contributes nothing instead of dragging the fused vector toward the zero
embedding. Gates are returned on every prediction — that is what the UI shows as
"why this verdict".

## The design constraint that matters

Most 2022-era phishing hosts no longer resolve, so a large fraction of rows have a
URL and nothing else. Masked modality dropout during training is **load-bearing**:
without it the gates collapse onto whichever modality is most often complete.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

NOTEBOOK_DIR = BACKEND / "notebooks"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifact(name, obj):
    """Persist a JSON-serialisable object and echo where it landed."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"saved -> {path}")
    return path

## 6.1 Integrity gate — never train on false availability

`_read_html` returns `None` for a missing file rather than raising, and
`_image_tensor` returns zeros. So a deleted artifact silently becomes
*mask says available, tensor is all zeros*. This is the exact failure mode the
project exists to prevent, so check it before anything else.

In [ ]:
from pathlib import Path
from app.preprocessing.multimodal_dataset import load_manifest

mf = load_manifest(BACKEND / "data" / "manifest.csv")

bad_html = [p for p in mf.loc[mf.html_ok == True, "html_path"] if not Path(str(p)).is_file()]
bad_shot = [p for p in mf.loc[mf.shot_ok == True, "shot_path"] if not Path(str(p)).is_file()]

show("integrity", {"html_missing": len(bad_html), "shot_missing": len(bad_shot)})

if bad_html or bad_shot:
    mf.loc[mf.html_path.isin(bad_html), "html_ok"] = False
    mf.loc[mf.shot_path.isin(bad_shot), "shot_ok"] = False
    print("repaired: availability flags set False for artifacts absent from disk")

show("availability after repair", {
    "html_ok": int((mf.html_ok == True).sum()),
    "shot_ok": int((mf.shot_ok == True).sum()),
    "rows": len(mf),
})

## 6.2 The confound — read before interpreting any fusion number

In [ ]:
hr("availability by label")
show("html_ok rate",  mf.groupby("label")["html_ok"].mean().round(4).to_dict())
show("shot_ok rate",  mf.groupby("label")["shot_ok"].mean().round(4).to_dict())

g = mf.groupby("label")["html_ok"].mean()
print(f"\navailability gap (phishing - legitimate) = {g[1] - g[0]:+.4f}")

### Mask-only baseline — the number any fusion gain must beat

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

y = (mf["label"] == 1).astype(int).values
h = (mf["html_ok"] == True).astype(int).values

print("a classifier that reads ONLY the availability mask:")
print(f"  F1        {f1_score(y, h):.4f}")
print(f"  precision {precision_score(y, h):.4f}")
print(f"  recall    {recall_score(y, h):.4f}")

print("""
This is not a detector. It exploits the fact that phishing pages were more likely to
still be live when crawled. Reporting fusion F1 without this baseline makes a
crawl-artefact look like a modelling win.""")

## 6.3 Tokenisers and scaler (fit on train only)

In [ ]:
from app.preprocessing.url_dataset import load_splits, subsample
from app.preprocessing.url_preprocessing import CharTokenizer
from app.preprocessing.url_features import FeatureScaler, extract_batch
from app.preprocessing.html_tokenizer import HTMLTextTokenizer

splits = load_splits(BACKEND / "data" / "splits")
train_sp = subsample(splits["train"], 20000, seed=SEED)

tok = CharTokenizer(max_length=256, min_count=5).fit(train_sp.urls)
scaler = FeatureScaler().fit(extract_batch(train_sp.urls))
show("url side", {"vocab_size": tok.vocab_size, "max_length": 256, "n_features": 30})

In [ ]:
from app.preprocessing.multimodal_dataset import _soup_of
from app.preprocessing.html_features import visible_text_of

html_texts = []
for p in mf.loc[(mf.split == "train") & (mf.html_ok == True), "html_path"].head(2000):
    try:
        html_texts.append(visible_text_of(_soup_of(
            Path(str(p)).read_text(encoding="utf-8", errors="replace")))[:4000])
    except OSError:
        continue

htok = HTMLTextTokenizer(vocab_size=4096, max_tokens=512, min_count=2).fit(html_texts)
show("html tokenizer", {"vocab_size": htok.size, "max_tokens": 512, "docs": len(html_texts)})

size_once = htok.size
htok.fit(html_texts[:5])
print(f"\nsecond fit: {size_once} -> {htok.size}")
print("""
HTMLTextTokenizer.fit APPENDS to `itos` rather than rebuilding it, so calling fit
twice on different corpora corrupts the vocabulary mapping. Here vocab_size=4096
capped it so the count did not visibly change — the risk is real, the demo is weak.
Call fit exactly once.""")

## 6.4 Build rows

In [ ]:
from app.preprocessing.multimodal_dataset import build_rows, MultiModalDataset, collate_multimodal

train_rows = build_rows(
    mf, tokenizer=tok, scaler=scaler, html_tokenizer=htok, splits=("train",)
)
print(f"built {len(train_rows)} rows")

r0 = train_rows[0]
show("row 0", {
    "row_id": r0.row_id, "label": r0.label,
    "has_html": r0.has_html, "has_vision": r0.has_vision,
    "mask": r0.mask(),
    "html_vec": tuple(r0.html_vec.shape),
    "url_chars": tuple(r0.url_chars.shape),   # 1-D per row, not (B, L)
    "url_mask": tuple(r0.url_mask.shape),
    "url_feats": tuple(r0.url_feats.shape),
})

# build_rows has an indentation bug: tokenizer set + scaler=None silently replaces
# url_mask with a zero-width tensor, so the attention mask is lost. Passing both
# keeps it at full width.
assert r0.url_mask.shape[0] == 256, f"url_mask destroyed: {tuple(r0.url_mask.shape)}"

broken = build_rows(mf, tokenizer=tok, scaler=None, splits=("train",))
show("build_rows indentation bug, demonstrated", {
    "tokenizer + scaler  -> url_mask": tuple(r0.url_mask.shape),
    "tokenizer only      -> url_mask": tuple(broken[0].url_mask.shape),
})
print("The `else` at multimodal_dataset.py:166 binds to `if scaler is not None`,")
print("so omitting the scaler silently discards the attention mask.")

## 6.5 HTML encoder — standalone forward pass

In [ ]:
import torch
from app.models.html_model import HTMLModalityModel

from app.preprocessing.html_features import N_HTML_FEATURES

html_model = HTMLModalityModel(
    vocab_size=htok.size,
    max_tokens=512,
    n_dom_features=N_HTML_FEATURES,
    embedding_out=128,
)
print(f"html params: {sum(p.numel() for p in html_model.parameters()):,}")
html_model.eval()

# Only rows that actually collected HTML carry token ids. Rows without HTML have
# zero-width token tensors, so mixing them into one batch would break the embedding
# lookup. Select rows with HTML for this encoder demo.
demo_rows = [r for r in train_rows if r.has_html][:64]
if not demo_rows:
    raise RuntimeError("No rows with HTML in the manifest. Run the collector first.")

demo_ds = MultiModalDataset(demo_rows, image_size=224)
batch = collate_multimodal([demo_ds[i] for i in range(8)])

with torch.no_grad():
    hout = html_model(batch["html_tokens"], batch["html_token_mask"], batch["html_vec"])
show("html output", {"embedding": tuple(hout.embedding.shape), "logit": tuple(hout.logit.shape)})
show("batch keys", {k: tuple(v.shape) for k, v in batch.items() if hasattr(v, "shape")})

## 6.6 Vision encoder

In [ ]:
from app.models.vision_model import VisionModalityModel

vision_model = VisionModalityModel(backbone="resnet18", pretrained=False, embedding_out=128)
print(f"vision params: {sum(p.numel() for p in vision_model.parameters()):,}")
print("NOTE pretrained=False above. Any report MUST state whether weights actually loaded.")
vision_model.eval()

with torch.no_grad():
    vout = vision_model(batch["image"])
show("vision output", {"embedding": tuple(vout.embedding.shape), "logit": tuple(vout.logit.shape)})

## 6.7 Freeze encoders, cache embeddings

Stage two trains **only** the fusion head on precomputed embeddings. The encoders are
frozen and set to `eval()` so dropout and batchnorm do not drift.

In [ ]:
loader = torch.utils.data.DataLoader(
    MultiModalDataset(train_rows[:512], image_size=224),
    batch_size=32, shuffle=False, collate_fn=collate_multimodal,
)

emb_html, emb_vis, masks, ys = [], [], [], []

html_model.eval(); vision_model.eval()
with torch.no_grad():
    for b in loader:
        if b["html_tokens"].numel() == 0:
            # No HTML anywhere in this batch: emit a zero embedding so the
            # availability mask, not the tensor, carries the meaning.
            emb_html.append(torch.zeros(b["label"].size(0), 128))
        else:
            emb_html.append(
                html_model(b["html_tokens"], b["html_token_mask"], b["html_vec"]).embedding
            )
        emb_vis.append(vision_model(b["image"]).embedding)
        masks.append(torch.stack([b["mask_url"], b["mask_html"], b["mask_vision"]], dim=1))
        ys.append(b["label"])

E_html = torch.cat(emb_html)
E_vis = torch.cat(emb_vis)
M = torch.cat(masks)
Y = torch.cat(ys)

show("cached", {
    "emb_html": tuple(E_html.shape),
    "emb_vision": tuple(E_vis.shape),
    "masks": tuple(M.shape),
    "labels": tuple(Y.shape),
})
print("\nmasks as url/html/vision availability, per row")
show("mask column means", {"url": round(float(M[:,0].mean()),4),
                            "html": round(float(M[:,1].mean()),4),
                            "vision": round(float(M[:,2].mean()),4)})

## 6.8 Fusion model

In [ ]:
from app.models.fusion_model import FusionModel

SHARED = 128
# URL embeddings come from the trained URL model. Here a placeholder of the correct
# width keeps the fusion wiring visible WITHOUT leaking the label through the URL
# embedding. Any score below is therefore attributable to the availability mask.
E_url = torch.randn(len(Y), SHARED)

fusion = FusionModel(shared_dim=SHARED, hidden_dims=(128, 64), modality_dropout=0.3)
print(f"fusion params: {sum(p.numel() for p in fusion.parameters()):,}")
print(f"modality_dropout = 0.3 (built into FusionModel.forward during training)")

with torch.no_grad():
    f_out = fusion(E_url, E_html, E_vis, M[:,0], M[:,1], M[:,2])

show("fusion output", {
    "logit": tuple(f_out.logit.shape),
    "gates": tuple(f_out.gates.shape),
    "modality_logits": {k: tuple(v.shape) for k, v in f_out.modality_logits.items()},
})

### Gates must renormalise over AVAILABLE modalities only

In [ ]:
with torch.no_grad():
    gates = fusion(E_url, E_html, E_vis, M[:,0], M[:,1], M[:,2]).gates

avail = M.bool()
per_row_sum = gates.sum(dim=1)

print("gate rows for rows where ALL modalities are available:")
full = avail.all(dim=1)
print("  ", [round(float(g), 4) for g in gates[full][0]])

print("\ngate rows where ONLY url is available:")
only_url = (M[:,0] == 1) & (M[:,1] == 0) & (M[:,2] == 0)
if only_url.sum():
    print("  ", [round(float(g), 4) for g in gates[only_url][0]])

hr("gate mass by availability pattern")
for name, sel in {
    "all three": avail.all(dim=1),
    "url+html":  (M[:,0]==1) & (M[:,1]==1) & (M[:,2]==0),
    "url only":  only_url,
}.items():
    if sel.sum():
        print(f"  {name:<10} n={int(sel.sum()):>4}  mean gate sum={per_row_sum[sel].mean():.4f}")

print("""
A missing modality gets ~zero weight, so the zero embedding does not pull the
fused vector around. This is the behaviour that makes partial inputs safe.""")

## 6.9 Train the fusion head

In [ ]:
from torch import nn

opt = torch.optim.AdamW(fusion.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

fusion.train()
N = len(Y)
running = 0.0
for step in range(300):
    idx = torch.randint(0, N, (64,))
    m = M[idx]
    out = fusion(E_url[idx], E_html[idx], E_vis[idx], m[:,0], m[:,1], m[:,2])
    loss = loss_fn(out.logit, Y[idx])
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(fusion.parameters(), 1.0)
    opt.step()
    running += float(loss)

print(f"mean training loss over 300 steps: {running/300:.4f}")

## 6.10 Evaluation — and why a perfect score here is a red flag

The encoders above are **untrained** and the URL embeddings are random placeholders.
So any near-perfect score below is **not** modelling success. It is the availability
confound leaking in, and reading it correctly matters more than the number.

In [ ]:
from app.utils.metrics import compute_binary_metrics

fusion.eval()
with torch.no_grad():
    probs = torch.sigmoid(fusion(E_url, E_html, E_vis, M[:,0], M[:,1], M[:,2]).logit).numpy()

y_true = Y.numpy()

hr("overall")
mt = compute_binary_metrics(y_true, probs)
show("metrics", {k: round(getattr(mt, k), 6) for k in
                 ("accuracy","precision","recall","f1","specificity","roc_auc","mcc")})

print("""
Read this carefully: the encoders are untrained and E_url is random noise, yet F1
may look perfect. The only signal available to the fusion head is the availability
mask, which is confounded with the label (section 6.2). It has learned
'HTML was collected -> phishing', which is a crawl artefact.""")

In [ ]:
# Demonstrate the confound directly: score using ONLY the availability mask.
from sklearn.metrics import f1_score

mask_score = M[:, 1].numpy()          # mask_html alone
print("mask_html alone as a classifier:")
print(f"  F1 {f1_score(y_true, mask_score):.4f}")

print("""
Compare that to the fusion F1 above. If fusion barely beats the mask alone, the
'multimodal gain' is the confound, not the modalities. This comparison is mandatory
before claiming fusion helps.""")

In [ ]:
print("per-modality subsets — the number that matters in practice:")
for name, sel in {
    "url only":  (M[:,0]==1) & (M[:,1]==0) & (M[:,2]==0),
    "url + html":(M[:,0]==1) & (M[:,1]==1) & (M[:,2]==0),
    "all three": avail.all(dim=1),
}.items():
    s = sel.numpy()
    if s.sum() < 5:
        print(f"  {name:<12} n={int(s.sum()):>4}  (too few to score)")
        continue
    ms = compute_binary_metrics(y_true[s], probs[s])
    print(f"  {name:<12} n={int(s.sum()):>4}  F1={ms.f1:.4f}  recall={ms.recall:.4f}")
    # Also show the mask-only score on the SAME subset, for a fair comparison.
    print(f"  {'':<12}   mask-only F1={f1_score(y_true[s], mask_score[s]):.4f} on the same rows")

print("""
If 'url only' collapses relative to 'all three', the fusion head has learned to
lean on modalities that are usually missing at inference. That gap is the honest
measurement of the design.""")

## 6.11 Persist

In [ ]:
OUT = ARTIFACT_DIR / "fusion_notebook.pt"
torch.save({
    "state_dict": fusion.state_dict(),
    "modality_dropout": 0.3,
    "n_rows_cached": int(N),
    "seed": SEED,
    "note": (
        "DEMONSTRATION ONLY. Encoders are untrained and url embeddings are random "
        "placeholders. Any score from this notebook is the availability confound, "
        "not a model result."
    ),
}, OUT)
show("saved", {"path": str(OUT), "size_kb": round(OUT.stat().st_size/1024, 1)})

## 6.12 Reporting checklist

Confirm every item before writing up a multimodal result.

In [ ]:
print("""[ ] html_ok / shot_ok match files actually on disk        (section 6.1)
[ ] the availability gap is stated as a limitation           (section 6.2)
[ ] fusion is compared against the MASK-ONLY baseline, not chance  (6.2)
[ ] per-modality subsets are reported, not just the pooled score (6.10)
[ ] pretrained backbone weights actually loaded, or it says random-init
[ ] the URL model number is framed as PhiUSIIL benchmark separability

Omitting the first two is how a multimodal phishing detector ends up claiming
credit for knowing which phishing sites happened to still be online.""")

> **Note.** These notebooks *call* the production modules in `app/` and
> `training/`. They do not reimplement the pipeline. The logic under test lives
> in exactly one place, so a notebook can never drift from what the CLI scripts
> and the API actually run.